# JADES DR5 (GOODS-S + GOODS-N) F150W — download & explore

Mirrors `data/survey/CEERS/ceers_cat.ipynb`: download the mosaics / segmaps / photometry
catalogs for both JADES fields (only if missing, for reproducibility), then sanity-check
units, pixel scale, segmap alignment, and the catalog columns used by the exporter.

JADES = NIRCam, **MJy/sr @ 30 mas** just like COSMOS-Web / CEERS, so cutouts need **no
flux conversion and no resampling** (unlike OutThere). The only twist: the data products
are split by field (GOODS-S / GOODS-N); `cutout_export_jades.py` merges them into a single
`image_index_jades_f150w.fits` with `tile` = field alias (`gds` / `gdn`).

In [ ]:
# Download JADES DR5 GOODS-S + GOODS-N products (mosaic, segmentation, photometry catalog).
# Only fetches files that are missing, so re-running is a no-op (reproducible).
import os

BASE = 'https://slate.ucsc.edu/~brant/jades-dr5'
FIELDS = {'GOODS-S': 'goods-s', 'GOODS-N': 'goods-n'}  # url dir -> file-name slug

for FIELD, slug in FIELDS.items():
    targets = {
        f'hlsp_jades_jwst_nircam_{slug}_f150w_v5.0_drz.fits':
            f'{BASE}/{FIELD}/hlsp/images/mosaics/hlsp_jades_jwst_nircam_{slug}_f150w_v5.0_drz.fits',
        f'hlsp_jades_jwst_nircam_{slug}_segmentation_v5.0_drz.fits':
            f'{BASE}/{FIELD}/hlsp/images/mosaics/hlsp_jades_jwst_nircam_{slug}_segmentation_v5.0_drz.fits',
        f'hlsp_jades_jwst_nircam_{slug}_photometry_v5.0_catalog.fits':
            f'{BASE}/{FIELD}/hlsp/catalogs/hlsp_jades_jwst_nircam_{slug}_photometry_v5.0_catalog.fits',
    }
    for fname, url in targets.items():
        if os.path.exists(fname):
            print(f'have   {fname}')
            continue
        print(f'fetch  {fname}')
        ret = os.system(f'wget -q --show-progress "{url}" -O "{fname}"')
        if ret != 0:
            os.remove(fname)  # don't leave a truncated file behind
            raise RuntimeError(f'download failed: {url}')
print('done')

In [ ]:
# Inspect the mosaic + segmap: extensions, units, pixel scale, shape.
from astropy.io import fits
from astropy.wcs import WCS
import numpy as np, warnings
warnings.simplefilter('ignore')

for slug in ('goods-s', 'goods-n'):
    mos = f'hlsp_jades_jwst_nircam_{slug}_f150w_v5.0_drz.fits'
    seg = f'hlsp_jades_jwst_nircam_{slug}_segmentation_v5.0_drz.fits'
    print(f'\n===== {slug} =====')
    with fits.open(mos) as h:
        h.info()
        sci = next(i for i, hd in enumerate(h) if hd.data is not None)
        hdr = h[sci].header
        w = WCS(hdr)
        pix_mas = np.sqrt(np.abs(np.linalg.det(w.pixel_scale_matrix))) * 3.6e6
        print(f'  SCI ext={sci}  BUNIT={hdr.get("BUNIT")}  pix={pix_mas:.3f} mas  shape={h[sci].data.shape}')
    with fits.open(seg) as h:
        segext = next(i for i, hd in enumerate(h) if hd.data is not None)
        print(f'  SEG ext={segext}  dtype={h[segext].data.dtype}  max_id={int(h[segext].data.max())}')

In [ ]:
# Inspect the photometry catalog: list extensions and the columns the exporter needs
# (ID / RA / DEC + F150W flux & error for the SNR cut, F150W_FLAG for quality).
from astropy.table import Table

for slug in ('goods-s', 'goods-n'):
    cat = f'hlsp_jades_jwst_nircam_{slug}_photometry_v5.0_catalog.fits'
    print(f'\n===== {slug} =====')
    with fits.open(cat) as h:
        for i, hd in enumerate(h):
            name = hd.header.get('EXTNAME', '')
            nrow = hd.header.get('NAXIS2', '')
            ncol = hd.header.get('TFIELDS', '')
            print(f'  [{i}] {name:16s} rows={nrow} cols={ncol}')

In [ ]:
# Visualize one random object cutout per field via the mosaic WCS (sanity-check positions).
import matplotlib.pyplot as plt
from astropy.coordinates import SkyCoord
import astropy.units as u

fig, axes = plt.subplots(1, 2, figsize=(9, 4.5))
for ax, slug in zip(axes, ('goods-s', 'goods-n')):
    mos = f'hlsp_jades_jwst_nircam_{slug}_f150w_v5.0_drz.fits'
    cat = f'hlsp_jades_jwst_nircam_{slug}_photometry_v5.0_catalog.fits'
    main = Table.read(cat, hdu=2)  # ID / RA / DEC live in the main (HDU 2) table
    with fits.open(mos) as h:
        sci = next(i for i, hd in enumerate(h) if hd.data is not None)
        w = WCS(h[sci].header)
        data = h[sci].data
        idx = np.random.randint(len(main))
        x, y = w.world_to_pixel(SkyCoord(main['RA'][idx], main['DEC'][idx], unit='deg'))
        s = 64
        cut = data[int(y) - s:int(y) + s, int(x) - s:int(x) + s]
    ax.imshow(np.arcsinh(cut / 0.02), origin='lower', cmap='gray')
    ax.set_title(f'{slug}  ID={main["ID"][idx]}')
plt.tight_layout(); plt.show()